**Student name:** Ghala Mohammed Alkahtani

In [ ]:
!pip install -q langgraph langchain langchain-google-genai langchain-chroma langchain-community langchain-text-splitters chromadb pydantic

In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Gemini API Key: ")

langsmith_key = getpass("LangSmith API Key : ")
if langsmith_key.strip():
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_API_KEY"] = langsmith_key
    os.environ["LANGCHAIN_PROJECT"] = "capstone-agentic-rag"
    print("LangSmith tracing enabled")
else:
    print("LangSmith tracing disabled")

Gemini API Key: ··········
LangSmith API Key : ··········
LangSmith tracing enabled


In [ ]:
from pathlib import Path

DATA_DIR = Path('data')
(DATA_DIR / 'product_docs').mkdir(parents=True, exist_ok=True)
(DATA_DIR / 'policies').mkdir(parents=True, exist_ok=True)

product_docs = {
  "getting_started.md": "# Getting Started with CloudSuite\n\nCloudSuite is a project management and collaboration platform. This guide covers\naccount setup, workspace creation, and inviting team members.\n\n## Creating a Workspace\n1. Sign in to your CloudSuite account.\n2. Click \"New Workspace\" and choose a name and region (US, EU, or APAC).\n3. Select a plan: Free, Team, or Enterprise.\n\n## Inviting Team Members\nWorkspace owners and admins can invite members from Settings > Members > Invite.\nInvited users receive an email link that expires after 7 days. Free plan\nworkspaces are limited to 5 members; Team plan workspaces support up to 50.\n\n## Integrations\nCloudSuite supports integrations with Slack, GitHub, and Google Calendar.\nIntegrations are configured per-workspace under Settings > Integrations, and\nrequire an admin role to enable.\n\n## API Access\nAPI keys can be generated under Settings > Developer > API Keys. Each key is\nscoped to a single workspace and can be revoked at any time. Rate limits are\n100 requests/minute on the Free plan and 1000 requests/minute on Team and\nEnterprise plans.\n",
  "billing.md": "# CloudSuite Billing Guide\n\n## Plans and Pricing\n- Free: $0/month, 5 members, 1GB storage.\n- Team: $12/user/month, up to 50 members, 100GB storage.\n- Enterprise: custom pricing, unlimited members, unlimited storage, SSO included.\n\n## Changing Plans\nWorkspace owners can upgrade or downgrade a plan from Settings > Billing > Plan.\nUpgrades take effect immediately and are prorated for the current billing cycle.\nDowngrades take effect at the start of the next billing cycle.\n\n## Payment Methods\nCloudSuite accepts credit cards and, for Enterprise customers, wire transfer or\ninvoicing with net-30 terms. Payment method can be updated under Settings >\nBilling > Payment Methods.\n\n## Failed Payments\nIf a payment fails, CloudSuite retries automatically after 3 days and again\nafter 7 days. After a third failed attempt, the workspace is downgraded to\nFree plan limits until payment succeeds, but no data is deleted.\n",
  "troubleshooting.md": "# Troubleshooting CloudSuite\n\n## Login Issues\nIf you cannot log in, first check that your email is verified (a verification\nlink is sent on signup). If you use SSO, contact your workspace admin, since\nonly admins can reset SSO configuration.\n\n## Sync Delays\nReal-time sync between the desktop and web app can lag by up to 30 seconds\nunder heavy load. If a delay exceeds 5 minutes, try signing out and back in;\nthis forces a full re-sync.\n\n## Integration Failures\nIf a Slack or GitHub integration stops posting updates, the connected\naccount's token may have expired. Re-authenticate the integration from\nSettings > Integrations > Reconnect.\n\n## Data Export\nWorkspace owners can export all data as a ZIP file (CSV + attachments) from\nSettings > Data > Export. Exports for workspaces over 10GB may take up to 24\nhours to prepare and are emailed as a download link valid for 48 hours.\n"
}

policies = {
  "refund_policy.md": "# Refund Policy\n\nCloudSuite offers a full refund within 14 days of the initial purchase of a\nTeam or Enterprise plan, no questions asked. Requests after 14 days are\nevaluated case-by-case by the billing team and are not guaranteed.\n\nRefunds for annual plans canceled after 14 days are prorated only for unused\nfull months remaining, minus a 10% administrative fee. Monthly plans are not\neligible for partial-period refunds after the 14-day window.\n\nRefund requests must be submitted via the Support portal with the workspace\nID and reason for cancellation. Approved refunds are processed within 5-10\nbusiness days to the original payment method.\n",
  "privacy_policy.md": "# Privacy Policy Summary\n\nCloudSuite stores customer data in the region selected at workspace creation\n(US, EU, or APAC) and does not transfer data across regions without explicit\ncustomer request.\n\nData is encrypted at rest (AES-256) and in transit (TLS 1.2+). Employee\naccess to customer data requires a support ticket and is logged; access logs\nare retained for 1 year.\n\nCustomers may request full account deletion at any time. Deletion removes all\nworkspace data within 30 days, except for records CloudSuite is legally\nrequired to retain (e.g., billing records, kept for 7 years per financial\nregulations).\n\nCloudSuite does not sell customer data to third parties. Data may be shared\nwith sub-processors (e.g., cloud hosting, email delivery) strictly to provide\nthe service, under signed data processing agreements.\n",
  "support_sla.md": "# Support SLA\n\n## Response Times\n- Free plan: best-effort, no guaranteed response time (community forum only).\n- Team plan: first response within 1 business day, via email support.\n- Enterprise plan: first response within 4 business hours, via email and\n  dedicated Slack channel. Critical outages get a 1-hour response target.\n\n## Severity Levels\n- Critical (Sev 1): workspace fully inaccessible or data loss risk.\n- High (Sev 2): major feature broken, no workaround.\n- Normal (Sev 3): minor bug or question, workaround available.\n\n## Escalation\nEnterprise customers can escalate a ticket to their assigned Customer Success\nManager if the SLA is missed. Escalated tickets are reviewed within 2 hours.\n"
}

for name, content in product_docs.items():
    (DATA_DIR / 'product_docs' / name).write_text(content)
for name, content in policies.items():
    (DATA_DIR / 'policies' / name).write_text(content)

print('Data files written:')
for p in sorted(DATA_DIR.rglob('*.md')):
    print(' -', p)

Data files written:
 - data/policies/privacy_policy.md
 - data/policies/refund_policy.md
 - data/policies/support_sla.md
 - data/product_docs/billing.md
 - data/product_docs/getting_started.md
 - data/product_docs/troubleshooting.md


In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

PERSIST_DIR = "chroma_db"
SOURCES = {"product_docs": DATA_DIR / "product_docs", "policies": DATA_DIR / "policies"}

def build_vectorstore(source_name, source_dir, embeddings):
    loader = DirectoryLoader(str(source_dir), glob="*.md", loader_cls=TextLoader)
    raw_docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500, chunk_overlap=75,
        separators=["\n## ", "\n# ", "\n\n", "\n", " "],
    )
    chunks = splitter.split_documents(raw_docs)
    for chunk in chunks:
        chunk.metadata["source_collection"] = source_name

    vs = Chroma.from_documents(
        documents=chunks, embedding=embeddings,
        collection_name=source_name, persist_directory=PERSIST_DIR,
    )
    print(f"[ingest] {source_name}: {len(raw_docs)} docs -> {len(chunks)} chunks embedded.")
    return vs

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstores = {name: build_vectorstore(name, path, embeddings) for name, path in SOURCES.items()}
print("Ingestion complete")

[ingest] product_docs: 3 docs -> 7 chunks embedded.
[ingest] policies: 3 docs -> 6 chunks embedded.
Ingestion complete


In [ ]:
from langgraph.store.memory import InMemoryStore

long_term_store = InMemoryStore()
NAMESPACE = "user_history"

def remember_query(user_id, query, source_used):
    existing = long_term_store.get((NAMESPACE,), user_id)
    history = existing.value["queries"] if existing else []
    history.append({"query": query, "source_used": source_used})
    long_term_store.put((NAMESPACE,), user_id, {"queries": history[-10:]})

def recall_user_history(user_id):
    existing = long_term_store.get((NAMESPACE,), user_id)
    return existing.value["queries"] if existing else []

print("Long-term memory store ready")

Long-term memory store ready


In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field

class RouteDecision(BaseModel):
    source: Literal["product_docs", "policies", "both"] = Field(
        description="Which knowledge source(s) are relevant to the question."
    )
    reasoning: str = Field(description="One short sentence justifying the choice.")

class Answer(BaseModel):
    answer: str
    confidence: float = Field(ge=0.0, le=1.0)
    sources_used: list

class EvaluationResult(BaseModel):
    score: float = Field(ge=0.0, le=1.0)
    feedback: Optional[str] = None
    grounded: bool = Field(description="True if the answer is actually supported by context.")

print("Schemas defined")

Schemas defined


In [33]:
from langchain_google_genai import ChatGoogleGenerativeAI

_llm = None

def get_llm():
    global _llm
    if _llm is None:
        _llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
    return _llm

def get_vectorstore(source_name):
    return vectorstores[source_name]

print("LLM/vectorstore getters ready")

LLM/vectorstore getters ready


In [ ]:
from langgraph.func import task
from langgraph.types import RetryPolicy

@task
def classify_query(question):
    """Routing workflow pattern: decide which knowledge source(s) to use."""
    structured_llm = get_llm().with_structured_output(RouteDecision)
    decision = structured_llm.invoke([
        ("system",
         "You route user questions to the right knowledge base(s). "
         "'product_docs' covers how CloudSuite works (setup, billing "
         "mechanics, troubleshooting). 'policies' covers legal/company "
         "policy questions (refunds, privacy, support SLAs). Choose "
         "'both' only if the question genuinely spans both."),
        ("user", question),
    ])
    return decision.model_dump()


@task(retry_policy=RetryPolicy(max_attempts=3, initial_interval=1.0, backoff_factor=2.0))
def retrieve_from_source(source_name, question, k=3):
    store = get_vectorstore(source_name)
    docs = store.similarity_search(question, k=k)
    return [{"text": d.page_content, "source": source_name} for d in docs]


@task
def generate_answer(question, context_chunks, feedback=None):
    context_text = "\n\n".join(f"[{c['source']}] {c['text']}" for c in context_chunks)
    system = (
        "Answer ONLY using the provided context. If the context does not "
        "contain the answer, say so explicitly instead of guessing. Report "
        "an honest confidence score."
    )
    if feedback:
        system += f"\n\nA previous attempt was rejected for this reason: {feedback}. Improve on that."

    structured_llm = get_llm().with_structured_output(Answer)
    result = structured_llm.invoke([
        ("system", system),
        ("user", f"Context:\n{context_text}\n\nQuestion: {question}"),
    ])
    return result.model_dump()


@task
def evaluate_answer(question, answer, context_chunks):
    """Evaluator-Optimizer workflow pattern: score the draft answer."""
    context_text = "\n\n".join(c["text"] for c in context_chunks)
    structured_llm = get_llm().with_structured_output(EvaluationResult)
    result = structured_llm.invoke([
        ("system",
         "You are a strict QA reviewer. Score 0-1 how well the answer "
         "is grounded in the context and actually answers the "
         "question. If score < 0.6, explain what's missing."),
        ("user", f"Question: {question}\n\nContext: {context_text}\n\nAnswer: {answer['answer']}"),
    ])
    return result.model_dump()

print("Tasks defined")

Tasks defined


In [ ]:
from langgraph.func import entrypoint
from langgraph.types import interrupt
from langgraph.checkpoint.memory import InMemorySaver

MAX_REGENERATE_ATTEMPTS = 2
LOW_CONFIDENCE_THRESHOLD = 0.6

@entrypoint(checkpointer=InMemorySaver())
def rag_agent(inputs):
    question = inputs["question"]
    user_id = inputs.get("user_id", "anonymous")

    history = recall_user_history(user_id)

    route = classify_query(question).result()
    sources = ["product_docs", "policies"] if route["source"] == "both" else [route["source"]]

    context_chunks = []
    for source_name in sources:
        context_chunks.extend(retrieve_from_source(source_name, question).result())

    if not context_chunks:
        user_rephrase = interrupt({
            "reason": "no_results",
            "message": f"No documents matched '{question}' in {sources}. "
                       "Please rephrase or confirm you want a best-effort answer anyway.",
        })
        if user_rephrase.get("action") == "rephrase":
            question = user_rephrase["new_question"]
            for source_name in sources:
                context_chunks.extend(retrieve_from_source(source_name, question).result())

    feedback = None
    answer = None
    evaluation = None
    for attempt in range(MAX_REGENERATE_ATTEMPTS + 1):
        answer = generate_answer(question, context_chunks, feedback).result()
        evaluation = evaluate_answer(question, answer, context_chunks).result()
        if evaluation["score"] >= LOW_CONFIDENCE_THRESHOLD or attempt == MAX_REGENERATE_ATTEMPTS:
            break
        feedback = evaluation["feedback"]

    if evaluation and (evaluation["score"] < LOW_CONFIDENCE_THRESHOLD or not evaluation["grounded"]):
        decision = interrupt({
            "reason": "low_confidence",
            "question": question,
            "draft_answer": answer["answer"],
            "score": evaluation["score"],
            "message": "This answer is low-confidence. Approve, edit, or reject?",
        })
        if decision.get("action") == "edit":
            answer["answer"] = decision["edited_answer"]
        elif decision.get("action") == "reject":
            answer["answer"] = "I don't have a reliable answer to this question yet."

    remember_query(user_id, question, source_used=route["source"])

    return {
        "answer": answer["answer"],
        "confidence": evaluation["score"] if evaluation else answer["confidence"],
        "route": route["source"],
        "sources_used": [c["source"] for c in context_chunks],
        "prior_topics_count": len(history),
    }

print("Agent graph compiled")

Agent graph compiled


In [ ]:
import uuid
from langgraph.types import Command

def run_query(question, user_id="demo_user", thread_id=None):
    thread_id = thread_id or str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    result = rag_agent.invoke({"question": question, "user_id": user_id}, config=config)

    while isinstance(result, dict) and "__interrupt__" in result:
        payload = result["__interrupt__"][0].value
        print("\n--- HUMAN INPUT NEEDED ---")
        print(payload["message"])

        if payload["reason"] == "no_results":
            choice = input("rephrase / skip: ")
            if choice.startswith("rephrase"):
                resume_value = {"action": "rephrase", "new_question": choice.split(" ", 1)[1]}
            else:
                resume_value = {"action": "skip"}
        elif payload["reason"] == "low_confidence":
            print(f"Draft answer: {payload['draft_answer']} (score={payload['score']:.2f})")
            choice = input("approve / edit / reject: ")
            if choice.startswith("edit"):
                resume_value = {"action": "edit", "edited_answer": choice.split(" ", 1)[1]}
            elif choice.startswith("reject"):
                resume_value = {"action": "reject"}
            else:
                resume_value = {"action": "approve"}
        else:
            resume_value = {"action": "skip"}

        result = rag_agent.invoke(Command(resume=resume_value), config=config)

    return result, thread_id

print("Runner ready")

Runner ready


In [ ]:
result, tid = run_query("How much does the Team plan cost per user?", user_id="student1")
result

{'answer': 'The Team plan costs $12/user/month.',
 'confidence': 1.0,
 'route': 'product_docs',
 'sources_used': ['product_docs', 'product_docs', 'product_docs'],
 'prior_topics_count': 2}

In [ ]:
result, tid = run_query("Can I get a refund after 20 days on a monthly plan?", user_id="student1", thread_id=tid)
result

{'answer': 'No, monthly plans are not eligible for partial-period refunds after the 14-day window.',
 'confidence': 1.0,
 'route': 'policies',
 'sources_used': ['policies', 'policies', 'policies'],
 'prior_topics_count': 3}

In [ ]:
result, tid = run_query("If I lose access due to a failed payment, what does the SLA say about getting support, and will I get a refund for the downtime?", user_id="student1")
result

{'answer': 'For the Free plan, support is best-effort with no guaranteed response time, and you can only use the community forum. Refunds are not mentioned in relation to failed payments or downtime. Refund requests for cancellations are processed within 5-10 business days to the original payment method, but this is separate from failed payment scenarios.',
 'confidence': 0.8,
 'route': 'both',
 'sources_used': ['product_docs',
  'product_docs',
  'product_docs',
  'policies',
  'policies',
  'policies'],
 'prior_topics_count': 4}

In [ ]:
recall_user_history("student1")

[{'query': 'How much does the Team plan cost per user?',
  'source_used': 'product_docs'},
 {'query': 'How much does the Team plan cost per user?',
  'source_used': 'product_docs'},
 {'query': 'Can I get a refund after 20 days on a monthly plan?',
  'source_used': 'policies'},
 {'query': 'If I lose access due to a failed payment, what does the SLA say about getting support, and will I get a refund for the downtime?',
  'source_used': 'both'}]

In [36]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("New Gemini API Key: ")

_llm = None

New Gemini API Key: ··········


In [37]:
LOW_CONFIDENCE_THRESHOLD = 1.1

result, tid_demo = run_query("How much does the Team plan cost per user?", user_id="demo_hitl")
result


--- HUMAN INPUT NEEDED ---
This answer is low-confidence. Approve, edit, or reject?
Draft answer: The Team plan costs $12/user/month. (score=1.00)
approve / edit / reject: approve


{'answer': 'The Team plan costs $12/user/month.',
 'confidence': 1.0,
 'route': 'product_docs',
 'sources_used': ['product_docs', 'product_docs', 'product_docs'],
 'prior_topics_count': 0}

In [38]:
LOW_CONFIDENCE_THRESHOLD = 0.6

In [39]:
result, tid_demo2 = run_query("What is the CEO's favorite football team?", user_id="demo_hitl")
result


--- HUMAN INPUT NEEDED ---
This answer is low-confidence. Approve, edit, or reject?
Draft answer: The provided context does not contain information about the CEO's favorite football team. (score=0.00)
approve / edit / reject: approve


{'answer': "The provided context does not contain information about the CEO's favorite football team.",
 'confidence': 0.0,
 'route': 'policies',
 'sources_used': ['policies', 'policies', 'policies'],
 'prior_topics_count': 2}

## Rubric Write-up

**Agent fundamentals:** `generate_answer` calls the LLM with real retrieved
context and returns structured output via the `Answer` Pydantic model, so
`confidence` and `sources_used` are reliable fields, not parsed text.

**Multi-agent / routing:** `classify_query` routes each question to
`product_docs`, `policies`, or `both`, and only the relevant source(s) are
queried afterward — matching Track C's routing pattern.

**RAG pipeline:** Load → split → embed → store is implemented per source
into two separate Chroma collections. We use **Hybrid RAG**: routing is
Agentic (the LLM picks the source), but retrieval within a source is 2-Step
(one `similarity_search` call), since the short documents don't need an
agentic retrieval loop.

**Context & state:** Short-term state is the `InMemorySaver` checkpointer,
keyed by `thread_id`. Long-term state is `long_term_store`, keyed by
`user_id`, and persists across sessions — shown when `recall_user_history`
returned all of `student1`'s past questions.

**Human-in-the-loop:** Two real `interrupt()` pauses were demonstrated: one
when retrieval found nothing relevant, and one forced by temporarily raising
`LOW_CONFIDENCE_THRESHOLD`. Both paused execution, waited for human input via
`Command(resume=...)`, then continued.

**LangGraph & error handling:** Built with `@task`/`@entrypoint`.
Implements transient retry (`RetryPolicy` on `retrieve_from_source`) and
user-fixable interrupt (no-results pause). LangSmith also captured
unhandled `ChatGoogleGenerativeAIError` bubble-ups from quota limits.

**Workflow pattern:** Two named patterns: **Routing** (`classify_query`)
and **Evaluator-Optimizer** (`evaluate_answer` scores drafts and can trigger
regeneration).

**LangSmith:** Tracing was enabled and reviewed. Several runs failed with
quota-exceeded errors, confirming why the retry policy matters, and latency
varied widely (2.5s–50s+) across similar questions, likely API-side load.